<img src="images/banner.png" style="width: 100%;">

In [1]:
from numpy.testing import assert_array_almost_equal

# Transformer Components

## 1 Scaled Dot-Product Attention

$$
\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V
$$

In [2]:
import numpy as np
from scipy.special import softmax

In [3]:
dim = 15

# Query, Key and Value with 15 embedding dimension
np.random.seed(1337)
query = np.random.random(size=(5, dim))
key = np.random.random(size=(10, dim))
value = np.random.random(size=(10, dim))

In [4]:
attention_score = softmax((query @ key.T) / np.sqrt(dim), axis=1) @ value

In [5]:
attention_score.shape

(5, 15)

In [6]:
def scaled_dot_product_attention(query, key, value):
    # b - batch size, d - embedding dimension
    scores = np.einsum("bqd,bkd->bqk", query, key)
    scores = softmax(scores / np.sqrt(key.shape[-1]), axis=-1)

    # v - embedding dimension of value
    return np.einsum("bqk,bkv->bqv", scores, value)

In [7]:
scaled_dot_product_attention(query[np.newaxis, :], key[np.newaxis, :], value[np.newaxis, :]).shape

(1, 5, 15)

In [8]:
assert_array_almost_equal(
    attention_score.flatten(),
    scaled_dot_product_attention(query[np.newaxis, :], key[np.newaxis, :], value[np.newaxis, :]).flatten()
)

## 2 Dense-Projections

In [9]:
from keras import layers

In [14]:
def dense_projections_attention(query, key, value, hidden_dim=20):
    # Dense projections
    query_dense = layers.Dense(hidden_dim)
    key_dense = layers.Dense(hidden_dim)
    value_dense = layers.Dense(hidden_dim)

    return scaled_dot_product_attention(
        query_dense(query),
        key_dense(key),
        value_dense(value)
    )

In [16]:
dense_projections_attention(query[np.newaxis, :], key[np.newaxis, :], value[np.newaxis, :]).shape

(1, 5, 20)

## 3 Multi-head Attention

<img src="images/multi-head-attention.svg" style="width: 30%;">

In [18]:
from keras import ops

In [19]:
def multi_head_attention(query, key, value, head_dim=20, num_heads=5):
    # Define heads
    query_dense = [layers.Dense(head_dim) for i in range(num_heads)]
    key_dense = [layers.Dense(head_dim) for i in range(num_heads)]
    value_dense = [layers.Dense(head_dim) for i in range(num_heads)]

    # Compute scaled dot product attention per head
    head_outputs = []
    for i in range(num_heads):
        head_output = scaled_dot_product_attention(
            query_dense[i](query),
            key_dense[i](key),
            value_dense[i](value)
        )
        head_outputs.append(head_output)

    # Concatenate output and project to another dense layer
    output_dense = layers.Dense(head_dim * num_heads)
    outputs = ops.concatenate(head_outputs, axis=-1)
    return output_dense(outputs)

In [21]:
multi_head_attention(query[np.newaxis, :], key[np.newaxis, :], value[np.newaxis, :]).shape

TensorShape([1, 5, 100])

<img src="images/banner-down.png" style="width: 100%;">